In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Check GPU
import torch
print(f"\n🖥️  GPU Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")
else:
    print("No GPU - but that's OK for preprocessing!")

print("\n✅ Setup complete!")

Mounted at /content/drive

🖥️  GPU Available: True
GPU Name: Tesla T4

✅ Setup complete!


In [ ]:
# Install packages (most are pre-installed in Colab)
!pip install opencv-python -q
!pip install pyyaml -q
!pip install tqdm -q

import cv2
import numpy as np
from pathlib import Path
from tqdm import tqdm
import yaml
import shutil
import matplotlib.pyplot as plt

print("✅ All packages loaded!")

✅ All packages loaded!


In [ ]:
from pathlib import Path

# Base directory
BASE_DIR = Path('/content/drive/MyDrive/CamouflageDetection')

# Input paths
TRAIN_IMAGES = BASE_DIR / '01_raw_data/train_images'
TRAIN_GT = BASE_DIR / '01_raw_data/train_gt'
TEST_IMAGES = BASE_DIR / '01_raw_data/test_images'
TEST_GT = BASE_DIR / '01_raw_data/test_gt'

# Output paths
OUTPUT_DIR = BASE_DIR / '02_dataset'
RESULTS_DIR = BASE_DIR / '04_results'

# Create output directories
OUTPUT_DIR.mkdir(exist_ok=True)
RESULTS_DIR.mkdir(exist_ok=True)

# Verify data exists
print("📂 Checking your data...")
print(f"Train images: {len(list(TRAIN_IMAGES.glob('*')))} files")
print(f"Train GT: {len(list(TRAIN_GT.glob('*')))} files")
print(f"Test images: {len(list(TEST_IMAGES.glob('*')))} files")
print(f"Test GT: {len(list(TEST_GT.glob('*')))} files")

# Expected: 748 train, 330 test
if len(list(TRAIN_IMAGES.glob('*'))) == 748:
    print("\n✅ Training data looks good!")
if len(list(TEST_IMAGES.glob('*'))) == 330:
    print("✅ Testing data looks good!")

📂 Checking your data...
Train images: 748 files
Train GT: 748 files
Test images: 330 files
Test GT: 330 files

✅ Training data looks good!
✅ Testing data looks good!


In [ ]:
def mask_to_yolo_bbox(mask_path):
    """
    Convert GT mask (black/white image) to YOLO bounding boxes

    Args:
        mask_path: Path to GT mask image

    Returns:
        List of bounding boxes in YOLO format: [class_id, x_center, y_center, width, height]
        All values normalized to 0-1
    """
    # Read mask as grayscale
    mask = cv2.imread(str(mask_path), cv2.IMREAD_GRAYSCALE)

    if mask is None:
        return []

    # Convert to binary (white = person, black = background)
    _, binary = cv2.threshold(mask, 127, 255, cv2.THRESH_BINARY)

    # Find contours (each contour = one person/object)
    contours, _ = cv2.findContours(binary, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    # Get image dimensions
    img_h, img_w = mask.shape

    bboxes = []

    for contour in contours:
        # Get bounding rectangle around contour
        x, y, w, h = cv2.boundingRect(contour)

        # Skip very small detections (noise)
        if w < 10 or h < 10:
            continue

        # Convert to YOLO format (normalized 0-1)
        x_center = (x + w/2) / img_w
        y_center = (y + h/2) / img_h
        width = w / img_w
        height = h / img_h

        # YOLO format: [class_id, x_center, y_center, width, height]
        # class_id = 0 for "camouflaged_person"
        bboxes.append([0, x_center, y_center, width, height])

    return bboxes


def process_split(images_dir, gt_dir, output_dir, split_name):
    """
    Process one dataset split (train or test)

    Args:
        images_dir: Folder with images
        gt_dir: Folder with GT masks
        output_dir: Where to save processed data
        split_name: 'train' or 'test'

    Returns:
        successful: Number of images processed
        skipped: Number of images skipped
    """
    images_dir = Path(images_dir)
    gt_dir = Path(gt_dir)
    output_dir = Path(output_dir)

    # Create output folders
    output_images = output_dir / split_name / 'images'
    output_labels = output_dir / split_name / 'labels'
    output_images.mkdir(parents=True, exist_ok=True)
    output_labels.mkdir(parents=True, exist_ok=True)

    # Get all image files
    image_files = sorted(list(images_dir.glob('*.jpg')) +
                        list(images_dir.glob('*.png')) +
                        list(images_dir.glob('*.jpeg')))

    print(f"\n📦 Processing {split_name} set: {len(image_files)} images")

    successful = 0
    skipped = 0

    # Process each image
    for img_path in tqdm(image_files, desc=f"{split_name}"):
        # Find corresponding GT mask
        # Try different possible naming conventions
        possible_gt_names = [
            img_path.stem + '.png',
            img_path.stem + '.jpg',
            img_path.stem + '_gt.png',
            img_path.stem + '_mask.png',
        ]

        gt_path = None
        for gt_name in possible_gt_names:
            temp_path = gt_dir / gt_name
            if temp_path.exists():
                gt_path = temp_path
                break

        if gt_path is None:
            # No GT found for this image
            skipped += 1
            continue

        # Convert mask to YOLO bounding boxes
        bboxes = mask_to_yolo_bbox(gt_path)

        if len(bboxes) == 0:
            # No valid bounding boxes found
            skipped += 1
            continue

        # Copy image to output
        shutil.copy(str(img_path), str(output_images / img_path.name))

        # Save YOLO labels
        label_file = output_labels / (img_path.stem + '.txt')
        with open(label_file, 'w') as f:
            for bbox in bboxes:
                # Write: class x_center y_center width height
                line = ' '.join([f'{x:.6f}' for x in bbox])
                f.write(line + '\n')

        successful += 1

    print(f"✅ {split_name}: Processed {successful} images, Skipped {skipped}")
    return successful, skipped


print("✅ Functions loaded!")

✅ Functions loaded!


In [ ]:
print("="*70)
print("🚀 Starting Data Preprocessing...")
print("="*70)

# Process training data
train_success, train_skip = process_split(
    images_dir=TRAIN_IMAGES,
    gt_dir=TRAIN_GT,
    output_dir=OUTPUT_DIR,
    split_name='train'
)

# Process testing data
test_success, test_skip = process_split(
    images_dir=TEST_IMAGES,
    gt_dir=TEST_GT,
    output_dir=OUTPUT_DIR,
    split_name='test'
)

# Create YOLO config file
config = {
    'path': str(OUTPUT_DIR),
    'train': 'train/images',
    'val': 'test/images',
    'test': 'test/images',
    'nc': 1,
    'names': ['camouflaged_person']
}

yaml_path = OUTPUT_DIR / 'data.yaml'
with open(yaml_path, 'w') as f:
    yaml.dump(config, f, default_flow_style=False)

print(f"\n✅ Config file created: {yaml_path}")

# Summary
print("\n" + "="*70)
print("📊 PREPROCESSING SUMMARY")
print("="*70)
print(f"Training: {train_success} images processed, {train_skip} skipped")
print(f"Testing: {test_success} images processed, {test_skip} skipped")
print(f"Total: {train_success + test_success} images ready for YOLO!")
print("="*70)

🚀 Starting Data Preprocessing...

📦 Processing train set: 748 images


train: 100%|██████████| 748/748 [08:46<00:00,  1.42it/s]


✅ train: Processed 748 images, Skipped 0

📦 Processing test set: 330 images


test: 100%|██████████| 330/330 [04:17<00:00,  1.28it/s]

✅ test: Processed 330 images, Skipped 0

✅ Config file created: /content/drive/MyDrive/CamouflageDetection/02_dataset/data.yaml

📊 PREPROCESSING SUMMARY
Training: 748 images processed, 0 skipped
Testing: 330 images processed, 0 skipped
Total: 1078 images ready for YOLO!


In [ ]:
def visualize_samples(dataset_dir, split='train', num_samples=5):
    """Show processed samples with bounding boxes"""

    img_dir = dataset_dir / split / 'images'
    label_dir = dataset_dir / split / 'labels'

    # Get sample images
    image_files = list(img_dir.glob('*.jpg'))
    if len(image_files) == 0:
        image_files = list(img_dir.glob('*.png'))

    if len(image_files) == 0:
        print(f"No images found in {img_dir}")
        return

    # Take first num_samples
    image_files = image_files[:num_samples]

    # Create figure
    fig, axes = plt.subplots(1, len(image_files), figsize=(4*len(image_files), 4))
    if len(image_files) == 1:
        axes = [axes]

    for idx, img_path in enumerate(image_files):
        # Read image
        img = cv2.imread(str(img_path))
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        h, w = img.shape[:2]

        # Read corresponding label
        label_path = label_dir / (img_path.stem + '.txt')

        if label_path.exists():
            with open(label_path, 'r') as f:
                for line in f:
                    # Parse YOLO format
                    parts = line.strip().split()
                    cls, x_c, y_c, width, height = map(float, parts)

                    # Convert to pixel coordinates
                    x1 = int((x_c - width/2) * w)
                    y1 = int((y_c - height/2) * h)
                    x2 = int((x_c + width/2) * w)
                    y2 = int((y_c + height/2) * h)

                    # Draw green bounding box
                    cv2.rectangle(img, (x1, y1), (x2, y2), (0, 255, 0), 3)
                    cv2.putText(img, 'person', (x1, y1-10),
                               cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)

        axes[idx].imshow(img)
        axes[idx].axis('off')
        axes[idx].set_title(img_path.name[:25], fontsize=10)

    plt.tight_layout()

    # Save
    save_path = RESULTS_DIR / f'{split}_samples.png'
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    print(f"✅ Saved: {save_path}")

    plt.show()


# Visualize training samples
print("\n📊 Visualizing training samples...")
visualize_samples(OUTPUT_DIR, 'train', num_samples=5)

# Visualize testing samples
print("\n📊 Visualizing testing samples...")
visualize_samples(OUTPUT_DIR, 'test', num_samples=5)

Output hidden; open in https://colab.research.google.com to view.

In [ ]:
# Final verification
print("\n" + "="*70)
print("🔍 DATASET VERIFICATION")
print("="*70)

train_images = list((OUTPUT_DIR / 'train/images').glob('*'))
train_labels = list((OUTPUT_DIR / 'train/labels').glob('*.txt'))
test_images = list((OUTPUT_DIR / 'test/images').glob('*'))
test_labels = list((OUTPUT_DIR / 'test/labels').glob('*.txt'))

print(f"Train images: {len(train_images)}")
print(f"Train labels: {len(train_labels)}")
print(f"Test images: {len(test_images)}")
print(f"Test labels: {len(test_labels)}")

# Check if counts match
if len(train_images) == len(train_labels):
    print("\n✅ Training data: Images and labels match!")
else:
    print("\n⚠️  Warning: Training images and labels don't match")

if len(test_images) == len(test_labels):
    print("✅ Testing data: Images and labels match!")
else:
    print("⚠️  Warning: Testing images and labels don't match")

# Check data.yaml exists
if (OUTPUT_DIR / 'data.yaml').exists():
    print("\n✅ data.yaml config file exists!")
    print(f"Location: {OUTPUT_DIR / 'data.yaml'}")

print("\n" + "="*70)
print("✅ PREPROCESSING COMPLETE!")
print("="*70)
print("\nYour dataset is ready for YOLO training!")
print("Next: Run Notebook 02 - Train YOLO")


🔍 DATASET VERIFICATION
Train images: 748
Train labels: 748
Test images: 330
Test labels: 330

✅ Training data: Images and labels match!
✅ Testing data: Images and labels match!

✅ data.yaml config file exists!
Location: /content/drive/MyDrive/CamouflageDetection/02_dataset/data.yaml

✅ PREPROCESSING COMPLETE!

Your dataset is ready for YOLO training!
Next: Run Notebook 02 - Train YOLO
